# FIRMS + Open-Meteo historical weather: weather-bearing candidate dataset

This is the direct training-data path: build the FIRMS/FEDS candidate rows, map each 1 km candidate cell to a compact weather location, retrieve ECMWF IFS hourly weather for that location and date, and join the hour at or before the row's prediction anchor. The result is a separate, uploadable weather-bearing dataset; the base candidate view stays immutable.

The weather anchor is the model prediction time (`anchor_at`), not a later time in the 12-hour label window. FIRMS acquisition timestamps remain the evidence used to seed and gate the candidate rows.

## Run order

1. Recollect FIRMS, FEDS, and terrain evidence for the requested source range, then build the positive view and base candidate view.
2. Pass that one completed candidate manifest to the weather backfill below. It calls Open-Meteo Historical Weather API with `models=ecmwf_ifs`, retains raw responses and the candidate-to-weather-tile mapping, and keeps the existing 600-location-unit/minute limiter.
3. Only a complete backfill can be joined and exported. A paused run is resumed from its partial manifest, which reuses already complete dates rather than re-requesting them. This prevents a partial weather collection from becoming an apparently complete upload.

These are retrospective historical-weather features for offline training. A live model must obtain equivalent current conditions or separately collected forecast-vintage inputs at inference time.

In [ ]:
from datetime import date, datetime, timezone
from pathlib import Path
import os

from wildfire_data.candidate_dataset import build_and_store_firms_candidate_dataset
from wildfire_data.collect_firms import collect_firms_range
from wildfire_data.open_meteo_historical import (
    DEFAULT_BATCH_SIZE,
    DEFAULT_MAX_CONSECUTIVE_RATE_LIMITS,
    DEFAULT_MAX_TILE_DISTANCE_METRES,
    DEFAULT_RATE_LIMIT_COOLDOWN_SECONDS,
    DEFAULT_REQUESTS_PER_MINUTE,
    backfill_open_meteo_historical_weather,
)
from wildfire_data.storage_budget import load_storage_budget
from wildfire_data.weather_candidate_dataset import (
    build_weather_candidate_dataset,
    export_weather_candidate_dataset_release,
)

DATA_ROOT = Path("data")
STORAGE_POLICY = load_storage_budget()
HISTORICAL_START = date(2026, 5, 11)
HISTORICAL_END = date(2026, 8, 22)
print(f"Archive root: {DATA_ROOT.resolve()} | UTC now: {datetime.now(timezone.utc):%Y-%m-%d %H:%M:%SZ}")

In [ ]:
# FIRMS needs one leading day for the 24-hour candidate-feature lookback.
# Recollect FEDS through 2026-08-23 separately so the last 12-hour label can be formed.
FIRMS_COLLECTION_START = date(2026, 5, 10)
FIRMS_COLLECTION_END = HISTORICAL_END
RECOLLECT_HISTORICAL_FIRMS = False

if RECOLLECT_HISTORICAL_FIRMS:
    api_key = (os.getenv("NASA_FIRMS_API_KEY") or os.getenv("MAP_KEY") or "").strip()
    if not api_key:
        raise RuntimeError("Set NASA_FIRMS_API_KEY (or MAP_KEY) before collecting FIRMS.")
    firms_result = collect_firms_range(
        str(DATA_ROOT),
        api_key=api_key,
        start_date=FIRMS_COLLECTION_START,
        end_date=FIRMS_COLLECTION_END,
        storage_budget=STORAGE_POLICY,
    )
    print(f"Archived {len(firms_result.responses):,} FIRMS responses; {firms_result.failed_count:,} ranges need retry.")
else:
    print("FIRMS recollection is disabled. Enable it only after setting the API key.")

In [ ]:
# Build the immutable no-weather candidate spine after FEDS labels, terrain, and the
# positive-only view have been rebuilt for HISTORICAL_START through HISTORICAL_END.
# Set POSITIVE_VIEW_MANIFEST to that completed positive-view manifest.
POSITIVE_VIEW_MANIFEST = None
BASE_CANDIDATE_MANIFEST = None  # Or set the path of an already completed candidate manifest.
BUILD_BASE_CANDIDATES = False

if BUILD_BASE_CANDIDATES:
    if POSITIVE_VIEW_MANIFEST is None:
        raise RuntimeError("Set POSITIVE_VIEW_MANIFEST before building candidates.")
    base_result = build_and_store_firms_candidate_dataset(
        DATA_ROOT,
        storage_budget=STORAGE_POLICY,
        start_date=HISTORICAL_START,
        end_date=HISTORICAL_END,
        split_start_date=HISTORICAL_START,
        split_end_date=HISTORICAL_END,
        positive_view_manifest=POSITIVE_VIEW_MANIFEST,
    )
    BASE_CANDIDATE_MANIFEST = base_result.manifest_path
    print(f"Built {base_result.candidate_row_count:,} base candidate rows: {BASE_CANDIDATE_MANIFEST}")
else:
    print("Base candidate build is disabled. Set BASE_CANDIDATE_MANIFEST to one completed view before backfill.")

In [ ]:
# Historical Open-Meteo weather: candidate location cover at its anchor hour.
# The default 10 km cover reduces calls; its mapping records the actual distance. Lower it for less spatial coalescing.
# The collector batches coordinates but charges every location to the rate limiter.
OPEN_METEO_MODEL = "ecmwf_ifs"
OPEN_METEO_REQUESTS_PER_MINUTE = DEFAULT_REQUESTS_PER_MINUTE  # Keep 600 unless the provider directs otherwise.
OPEN_METEO_BATCH_SIZE = DEFAULT_BATCH_SIZE
MAX_TILE_DISTANCE_M = DEFAULT_MAX_TILE_DISTANCE_METRES
RATE_LIMIT_COOLDOWN_SECONDS = DEFAULT_RATE_LIMIT_COOLDOWN_SECONDS
MAX_CONSECUTIVE_429S = DEFAULT_MAX_CONSECUTIVE_RATE_LIMITS
RESUME_WEATHER_BACKFILL_MANIFEST = None  # Set this to a partial manifest after a rate-limit or capacity pause.
BACKFILL_HISTORICAL_WEATHER = False
WEATHER_BACKFILL_MANIFEST = None  # Or set a completed backfill manifest before the join cell.

if BACKFILL_HISTORICAL_WEATHER:
    if BASE_CANDIDATE_MANIFEST is None:
        raise RuntimeError("Set BASE_CANDIDATE_MANIFEST; weather must be tied to one completed candidate view.")
    backfill_result = backfill_open_meteo_historical_weather(
        DATA_ROOT,
        storage_policy=STORAGE_POLICY,
        candidate_manifest=BASE_CANDIDATE_MANIFEST,
        start_date=HISTORICAL_START,
        end_date=HISTORICAL_END,
        resume_manifest=RESUME_WEATHER_BACKFILL_MANIFEST,
        model=OPEN_METEO_MODEL,
        max_tile_distance_m=MAX_TILE_DISTANCE_M,
        requests_per_minute=OPEN_METEO_REQUESTS_PER_MINUTE,
        batch_size=OPEN_METEO_BATCH_SIZE,
        rate_limit_cooldown_seconds=RATE_LIMIT_COOLDOWN_SECONDS,
        max_consecutive_rate_limits=MAX_CONSECUTIVE_429S,
    )
    WEATHER_BACKFILL_MANIFEST = backfill_result.manifest_path
    print(f"Weather backfill complete={backfill_result.complete}; manifest: {WEATHER_BACKFILL_MANIFEST}")
    if not backfill_result.complete:
        raise RuntimeError("Weather backfill paused or failed. Do not join it; set RESUME_WEATHER_BACKFILL_MANIFEST to this manifest and rerun the cell.")
else:
    print("Historical weather backfill is disabled. The 600-location-unit/minute limiter remains the default.")

In [ ]:
# Join only a complete backfill, then create a portable upload directory.
BUILD_WEATHER_DATASET = False
WEATHER_CANDIDATE_MANIFEST = None  # Or set a completed weather candidate manifest before export.
EXPORT_WEATHER_RELEASE = False
WEATHER_RELEASE_DIRECTORY = Path("releases/wildfire-spread-firms-feds-weather-2026-05-11_to_2026-08-22")

if BUILD_WEATHER_DATASET:
    if BASE_CANDIDATE_MANIFEST is None or WEATHER_BACKFILL_MANIFEST is None:
        raise RuntimeError("Set the matching base-candidate and complete weather-backfill manifests first.")
    weather_dataset_result = build_weather_candidate_dataset(
        DATA_ROOT,
        storage_budget=STORAGE_POLICY,
        candidate_manifest=BASE_CANDIDATE_MANIFEST,
        weather_backfill_manifest=WEATHER_BACKFILL_MANIFEST,
    )
    WEATHER_CANDIDATE_MANIFEST = weather_dataset_result.manifest_path
    print(f"Built {weather_dataset_result.candidate_row_count:,} weather-bearing rows: {WEATHER_CANDIDATE_MANIFEST}")

if EXPORT_WEATHER_RELEASE:
    if WEATHER_CANDIDATE_MANIFEST is None:
        raise RuntimeError("Set WEATHER_CANDIDATE_MANIFEST before export.")
    release = export_weather_candidate_dataset_release(
        DATA_ROOT,
        WEATHER_RELEASE_DIRECTORY,
        weather_candidate_manifest=WEATHER_CANDIDATE_MANIFEST,
    )
    print(f"Exported {release.candidate_row_count:,} weather-bearing rows to {release.directory}")

## Optional forward forecast capture

This is separate from the retrospective training backfill. Run it only while a forecast is operational, name the exact model run yourself, and keep its resulting issued-forecast features out of the historical-analysis upload.

In [ ]:
# Forward-only operational experiment: do not use this to recreate historical forecast availability.
# It reuses the 600 location-unit/minute rate limit and records successful-response availability.
import pandas as pd
from wildfire_data.forecast_tile_planning import iter_normalized_firms_detections
from wildfire_data.open_meteo_single_run import (
    DEFAULT_CANDIDATE_RADIUS_CELLS,
    DEFAULT_FORECAST_HORIZON_HOURS,
    capture_open_meteo_single_run,
    plan_firms_candidate_weather_tiles,
)

CAPTURE_FORWARD_FORECAST = False
FORWARD_FIRMS_START = datetime.now(timezone.utc).date()
FORWARD_FIRMS_END = FORWARD_FIRMS_START
FORWARD_MODEL = "ecmwf_ifs"
FORWARD_MODEL_RUN_AT = None  # Example: "2026-08-26T12:00:00Z" after that named run is accessible.

if CAPTURE_FORWARD_FORECAST:
    if FORWARD_MODEL_RUN_AT is None:
        raise RuntimeError("Set FORWARD_MODEL_RUN_AT to an explicit, already available UTC run.")
    forward_firms = pd.DataFrame(
        {
            "detection_id": record["detection_id"],
            "latitude": record["latitude"],
            "longitude": record["longitude"],
            "acquired_at": record["acquired_at"],
            "raw_artifact_id": record.get("provenance", {}).get("raw_artifact_id"),
        }
        for record in iter_normalized_firms_detections(
            DATA_ROOT, start_date=FORWARD_FIRMS_START, end_date=FORWARD_FIRMS_END
        )
    )
    if forward_firms.empty:
        raise RuntimeError("No normalized FIRMS detections exist for the forward window.")
    forward_plan = plan_firms_candidate_weather_tiles(
        forward_firms, candidate_radius_cells=DEFAULT_CANDIDATE_RADIUS_CELLS
    )
    forward_capture = capture_open_meteo_single_run(
        DATA_ROOT,
        forward_plan,
        model=FORWARD_MODEL,
        model_run_at=FORWARD_MODEL_RUN_AT,
        forecast_horizon_hours=DEFAULT_FORECAST_HORIZON_HOURS,
        storage_policy=STORAGE_POLICY,
        requests_per_minute=OPEN_METEO_REQUESTS_PER_MINUTE,
        batch_size=OPEN_METEO_BATCH_SIZE,
        rate_limit_cooldown_seconds=RATE_LIMIT_COOLDOWN_SECONDS,
        max_consecutive_rate_limits=MAX_CONSECUTIVE_429S,
    )
    print(f"Forward capture: {forward_capture.captured_tile_count:,}/{forward_capture.planned_tile_count:,} tiles; paused={forward_capture.paused_for_rate_limit}.")
else:
    print("Forward forecast capture is disabled.")

## What the upload contains

Each weather-bearing candidate row has `weather_temperature_2m`, `weather_relative_humidity_2m`, `weather_precipitation`, `weather_wind_u_10m`, and `weather_wind_v_10m`, plus the model, mapped weather tile, raw artifact IDs, and `weather_observed_at`. The join refuses a missing field, another row's tile, a later weather hour, or a partial backfill. ECMWF weather is coarser than the 1 km training grid; with the default cover a mapped request location may additionally be up to 10 km from a candidate centre, and the candidate-to-request distance plus returned grid location are retained.

For a live prediction service, collect the same features at the live prediction time. Open-Meteo Single Runs can remain a separate, forward-only forecast-vintage experiment; it must not be mixed with this retrospective training dataset.